# YOLO OBB Detections → CrochetPARADE Code Generator

**Purpose**: Take a crochet chart image, run YOLO OBB detection, and generate
valid, compilable [CrochetPARADE](https://www.crochetparade.org/) code that
reproduces the chart pattern.

**Pipeline**: Image → YOLO OBB → row grouping → stitch-graph analysis → CrochetPARADE output

**CrochetPARADE language reference**:
- Each line = one row/round
- Stitches: `ch`, `sc`, `hdc`, `dc`, `tr`, `dtr`, `ss` (slip stitch), `sk` (skip)
- Multipliers: `5ch`, `3*[sc,dc]`
- Turns: `turn` (must be last on a line)
- Fans/shells: `5dc` into one stitch (using `@` attachment when needed)
- Increases: `sc2inc` (2 sc into one stitch)
- Attachments: `@[row,stitch]`, `@[@+offset]`
- Labels: `sc.A`, `dc@A`
- Definitions: `DEF: name=stitch_sequence`
- Comments: `# text`

**8-class detection system**: chain(0), double(1), enseble\_chain(2), fan(3),
half\_double(4), noise(5), single(6), treble(7)


In [1]:
import cv2 as cv
import numpy as np
import math
import os
import sys
from dataclasses import dataclass, field
from collections import Counter
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

# ──── Paths (adjust for your environment) ────
# For Google Colab:
#   PROJECT_DIR = "/content/drive/MyDrive/Crochet_data"
#   MODEL_PATH  = "/content/drive/MyDrive/Crochet_data/runs/obb_train/weights/best.pt"
# For local:
PROJECT_DIR = os.path.abspath(".")
MODEL_PATH  = os.path.join(PROJECT_DIR, "runs", "obb", "train6", "weights", "best.pt")

# ──── Test image path (change to your chart image) ────
TEST_IMAGE  = os.path.join(PROJECT_DIR, "data", "raw", "img", "1.png")

# ──── Class mapping (matches project-11 classes.txt) ────
CLASS_MAP = {
    "enseble_chain":  0,
    "noise":          1,
    "fan":            2,
    "half_double":    3,
    "treble":         4,
    "chain":          5,
    "double":         6,
    "single":         7,
}

# CrochetPARADE stitch abbreviations for each YOLO class
YOLO_TO_PARADE = {
    "chain":         "ch",
    "double":        "dc",
    "enseble_chain": None,   # handled specially (turning chains / foundation)
    "fan":           None,   # handled specially (expands to multiple dc)
    "half_double":   "hdc",
    "noise":         None,   # skip — annotations, not stitches
    "single":        "sc",
    "treble":        "tr",
}

# Fan expansion: how many dc spokes a detected fan typically has
FAN_DEFAULT_SPOKES = 5

print(f"Project dir: {PROJECT_DIR}")
print(f"Model path:  {MODEL_PATH}")
print(f"Test image:  {TEST_IMAGE}")


Project dir: /Users/elevchenko/Documents/DataScience/Crochet
Model path:  /Users/elevchenko/Documents/DataScience/Crochet/runs/obb/train6/weights/best.pt
Test image:  /Users/elevchenko/Documents/DataScience/Crochet/data/raw/img/1.png


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Detection Infrastructure (same as util/tiler.py)
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class Detection:
    corners: np.ndarray   # (4, 2), float32, original image px
    cls_id: int
    cls_name: str
    confidence: float


def _bbox(corners):
    x1, y1 = corners.min(axis=0)
    x2, y2 = corners.max(axis=0)
    return x1, y1, x2, y2


def _rect_iou(a, b):
    ax1, ay1, ax2, ay2 = _bbox(a)
    bx1, by1, bx2, by2 = _bbox(b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def _nms(detections, iou_threshold):
    detections = sorted(detections, key=lambda d: d.confidence, reverse=True)
    kept = []
    while detections:
        best = detections.pop(0)
        kept.append(best)
        detections = [d for d in detections
                      if d.cls_id != best.cls_id
                      or _rect_iou(best.corners, d.corners) < iou_threshold]
    return kept


def _tile_starts(length, tile_size, stride):
    starts = list(range(0, length - tile_size, stride))
    if not starts or starts[-1] + tile_size < length:
        starts.append(max(0, length - tile_size))
    return starts


def _infer_tile(model, tile, x_offset, y_offset, conf):
    res = model.predict(tile, conf=conf, verbose=False)[0]
    dets = []
    for box in res.obb:
        corners = box.xyxyxyxy.cpu().numpy().reshape(4, 2).astype(np.float32)
        corners[:, 0] += x_offset
        corners[:, 1] += y_offset
        cls_id = int(box.cls[0])
        dets.append(Detection(corners, cls_id, res.names[cls_id], float(box.conf[0])))
    return dets


def predict_tiled(model, image, tile_size=640, overlap=0.2, conf=0.25,
                  iou_threshold=0.5):
    h, w = image.shape[:2]
    if h <= tile_size and w <= tile_size:
        return _infer_tile(model, image, 0, 0, conf)
    stride = max(1, int(tile_size * (1 - overlap)))
    all_dets = []
    for y1 in _tile_starts(h, tile_size, stride):
        for x1 in _tile_starts(w, tile_size, stride):
            tile = image[y1:min(y1 + tile_size, h), x1:min(x1 + tile_size, w)]
            all_dets.extend(_infer_tile(model, tile, x1, y1, conf))
    return _nms(all_dets, iou_threshold)


def estimate_stitch_size(model, image, tile_size=640, conf=0.15):
    h, w = image.shape[:2]
    scale = tile_size / max(h, w)
    if scale < 1.0:
        small = cv.resize(image, (int(w * scale), int(h * scale)))
    else:
        small, scale = image, 1.0
    res = model.predict(small, conf=conf, verbose=False)[0]
    if len(res.obb) == 0:
        return None
    sizes = []
    for box in res.obb:
        corners = box.xyxyxyxy.cpu().numpy().reshape(4, 2)
        side_a = np.linalg.norm(corners[0] - corners[1])
        side_b = np.linalg.norm(corners[1] - corners[2])
        sizes.append(min(side_a, side_b))
    return float(np.median(sizes)) / scale


def predict_adaptive(model, image, target_stitch_px=100, tile_size=640,
                     overlap=0.25, conf=0.25, iou_threshold=0.45):
    h, w = image.shape[:2]
    estimated_h = estimate_stitch_size(model, image, tile_size=tile_size,
                                       conf=min(conf, 0.15))
    if estimated_h is None or estimated_h <= 0:
        return predict_tiled(model, image, tile_size=tile_size, overlap=overlap,
                             conf=conf, iou_threshold=iou_threshold)
    effective_tile = int(tile_size * estimated_h / target_stitch_px)
    effective_tile = max(effective_tile, 64)
    if effective_tile >= max(h, w):
        return _infer_tile(model, image, 0, 0, conf)
    return predict_tiled(model, image, tile_size=effective_tile, overlap=overlap,
                         conf=conf, iou_threshold=iou_threshold)

print("Detection infrastructure loaded.")


Detection infrastructure loaded.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# Row Grouping & Geometry Extraction
# ═══════════════════════════════════════════════════════════════════════════════

def det_geometry(det):
    """Extract center, width, height, angle from a Detection."""
    corners = det.corners
    center = corners.mean(axis=0)
    w = float(np.linalg.norm(corners[0] - corners[1]))
    h = float(np.linalg.norm(corners[0] - corners[3]))
    angle = float(np.degrees(np.arctan2(
        corners[1, 1] - corners[0, 1],
        corners[1, 0] - corners[0, 0]
    )))
    return {
        "cx": float(center[0]), "cy": float(center[1]),
        "w": w, "h": h, "angle": angle,
    }


def group_into_rows(detections, img_h):
    """Cluster detections into rows by Y-coordinate proximity.
    Returns list of rows, each row sorted left-to-right.
    Rows are ordered bottom-to-top (row 0 = foundation, closest to bottom)."""
    if not detections:
        return []

    # Filter out noise
    dets = [d for d in detections if d.cls_name != "noise"]

    dets_with_cy = [(d, d.corners.mean(axis=0)[1]) for d in dets]
    dets_with_cy.sort(key=lambda x: x[1])

    # Estimate row gap threshold from median stitch height
    heights = []
    for d, _ in dets_with_cy:
        g = det_geometry(d)
        heights.append(max(g["w"], g["h"]))
    median_h = sorted(heights)[len(heights) // 2] if heights else 30
    row_gap = median_h * 0.45

    rows = []
    current_row = [dets_with_cy[0][0]]
    current_cy = dets_with_cy[0][1]

    for det, cy in dets_with_cy[1:]:
        if abs(cy - current_cy) > row_gap:
            rows.append(current_row)
            current_row = [det]
            current_cy = cy
        else:
            current_row.append(det)
            current_cy = current_cy * 0.8 + cy * 0.2

    if current_row:
        rows.append(current_row)

    # Sort each row left-to-right
    for row in rows:
        row.sort(key=lambda d: d.corners.mean(axis=0)[0])

    # Reverse so row 0 = bottom (foundation), row N = top
    rows.reverse()

    return rows


print("Row grouping ready.")


Row grouping ready.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Stitch Analysis & Pattern Recognition
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class RowAnalysis:
    """Analysis of a single row's stitch content."""
    row_idx: int
    detections: list
    stitch_types: list          # list of cls_name for each stitch
    stitch_positions: list      # list of (cx, cy) for each stitch
    is_chain_row: bool          # True if row is predominantly chains
    is_turning_chain: bool      # True if row is a short turning chain segment
    has_fans: bool              # True if row contains fan detections
    fan_positions: list         # indices of fan stitches in this row
    estimated_direction: str    # "ltr" or "rtl" (even/odd row alternation)


def analyze_row(row_dets, row_idx, prev_direction="ltr"):
    """Analyze a row of detections to determine structure."""
    types = [d.cls_name for d in row_dets]
    positions = [tuple(d.corners.mean(axis=0)) for d in row_dets]

    n = len(types)
    chain_count = types.count("chain")
    is_chain_row = (chain_count / max(n, 1)) > 0.7
    is_turning_chain = (chain_count >= 1 and n <= 5 and
                        all(t in ("chain", "enseble_chain") for t in types))

    has_fans = "fan" in types
    fan_positions = [i for i, t in enumerate(types) if t == "fan"]

    # Alternating direction: row 0 = ltr, row 1 = rtl, etc.
    direction = "ltr" if row_idx % 2 == 0 else "rtl"

    return RowAnalysis(
        row_idx=row_idx,
        detections=row_dets,
        stitch_types=types,
        stitch_positions=positions,
        is_chain_row=is_chain_row,
        is_turning_chain=is_turning_chain,
        has_fans=has_fans,
        fan_positions=fan_positions,
        estimated_direction=direction,
    )


def detect_v_stitches(row_dets):
    """Detect V-stitch pairs: two adjacent 'double' detections that are
    angled toward each other (one tilted left, one tilted right).
    Returns list of (left_idx, right_idx) pairs."""
    v_pairs = []
    i = 0
    while i < len(row_dets) - 1:
        d1 = row_dets[i]
        d2 = row_dets[i + 1]
        if d1.cls_name == "double" and d2.cls_name == "double":
            g1 = det_geometry(d1)
            g2 = det_geometry(d2)
            # V-stitch: left arm tilted clockwise (+angle), right arm counter-clockwise (-angle)
            # Check if they're close together horizontally and angled oppositely
            dx = abs(g1["cx"] - g2["cx"])
            avg_w = (g1["w"] + g2["w"]) / 2
            if dx < avg_w * 2.0:
                # Check angular opposition (one positive, one negative)
                if (g1["angle"] > 5 and g2["angle"] < -5) or \
                   (g1["angle"] < -5 and g2["angle"] > 5):
                    v_pairs.append((i, i + 1))
                    i += 2
                    continue
        i += 1
    return v_pairs


def detect_repeat_pattern(stitch_types):
    """Detect repeating stitch pattern within a row.
    Returns (repeat_unit, count) or None if no clean repeat found."""
    n = len(stitch_types)
    if n < 4:
        return None

    # Try repeat lengths from 2 to n//2
    for unit_len in range(2, n // 2 + 1):
        unit = stitch_types[:unit_len]
        count = 0
        for start in range(0, n, unit_len):
            end = min(start + unit_len, n)
            if stitch_types[start:end] == unit:
                count += 1
            else:
                break
        if count >= 2 and count * unit_len >= n - unit_len:
            # Check if remainder is a prefix of the unit
            remainder = stitch_types[count * unit_len:]
            if not remainder or remainder == unit[:len(remainder)]:
                return (unit, count, remainder)

    return None


print("Pattern analysis ready.")


Pattern analysis ready.


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# CrochetPARADE Code Generation
# ═══════════════════════════════════════════════════════════════════════════════

def stitch_to_parade(cls_name, count=1):
    """Convert a YOLO class name to CrochetPARADE stitch notation."""
    abbr = YOLO_TO_PARADE.get(cls_name)
    if abbr is None:
        return None  # handled specially (fan, enseble_chain, noise)
    if count > 1:
        return f"{count}{abbr}"
    return abbr


def compress_stitch_sequence(stitches):
    """Compress a list of stitch abbreviations by run-length encoding.
    E.g., ['sc','sc','sc','dc','dc'] -> ['3sc','2dc']"""
    if not stitches:
        return []
    compressed = []
    current = stitches[0]
    count = 1
    for s in stitches[1:]:
        if s == current:
            count += 1
        else:
            compressed.append(f"{count}{current}" if count > 1 else current)
            current = s
            count = 1
    compressed.append(f"{count}{current}" if count > 1 else current)
    return compressed


def row_to_parade_stitches(analysis, prev_row_analysis=None):
    """Convert a RowAnalysis into a list of CrochetPARADE stitch tokens."""
    tokens = []
    dets = analysis.detections
    types = analysis.stitch_types

    # Detect V-stitch pairs
    v_pairs = detect_v_stitches(dets)
    v_indices = set()
    for l, r in v_pairs:
        v_indices.add(l)
        v_indices.add(r)

    i = 0
    while i < len(types):
        cls_name = types[i]

        # Skip noise
        if cls_name == "noise":
            i += 1
            continue

        # Handle ensemble chain → turning chain
        if cls_name == "enseble_chain":
            # Count chain stitches in the ensemble
            g = det_geometry(dets[i])
            n_chains = max(1, round(g["h"] / max(g["w"] * 0.6, 8)))
            tokens.append(f"{n_chains}ch")
            i += 1
            continue

        # Handle fan → expand to Ndc
        if cls_name == "fan":
            tokens.append(f"{FAN_DEFAULT_SPOKES}dc")
            i += 1
            continue

        # Handle V-stitch pairs
        if i in v_indices:
            # Find the pair
            for l, r in v_pairs:
                if i == l:
                    tokens.extend(["dc", "ch", "dc"])
                    i = r + 1
                    break
            else:
                # Not found as left member, might be right of pair already processed
                abbr = YOLO_TO_PARADE.get(cls_name)
                if abbr:
                    tokens.append(abbr)
                i += 1
            continue

        # Standard stitch
        abbr = YOLO_TO_PARADE.get(cls_name)
        if abbr:
            tokens.append(abbr)

        i += 1

    return tokens


def generate_parade_row(analysis, row_num, total_rows, prev_analysis=None):
    """Generate a single CrochetPARADE row line."""
    tokens = row_to_parade_stitches(analysis, prev_analysis)

    if not tokens:
        return None

    # Try to compress with run-length encoding
    compressed = compress_stitch_sequence(tokens)

    # Try to detect repeating pattern
    repeat_result = detect_repeat_pattern(tokens)
    if repeat_result:
        unit, count, remainder = repeat_result
        unit_compressed = compress_stitch_sequence(unit)
        unit_str = ",".join(unit_compressed)
        if len(unit) > 1:
            line = f"{count}*[{unit_str}]"
        else:
            line = f"{count}{unit_str}"
        if remainder:
            rem_compressed = compress_stitch_sequence(remainder)
            line += "," + ",".join(rem_compressed)
    else:
        line = ",".join(compressed)

    # Add turn for flat work (not the last row)
    # In CrochetPARADE, turn must be the last element on a line
    if row_num < total_rows - 1 and not analysis.is_chain_row:
        line += ",turn"

    return line


def build_foundation_chain(first_row_analysis):
    """Generate foundation chain line from the first row stitch count."""
    n_stitches = len(first_row_analysis.stitch_types)
    # Foundation chain is typically stitch_count + 1 (for turning chain)
    # but for chain rows, use the actual count
    if first_row_analysis.is_chain_row:
        return f"{n_stitches}ch"
    else:
        # Standard: chain = number of stitches + turning chain
        # For sc rows: +1, for dc rows: +3, for hdc: +2, for tr: +4
        types = first_row_analysis.stitch_types
        dominant = Counter(t for t in types if t != "noise").most_common(1)
        if dominant:
            dom_type = dominant[0][0]
            extra = {"chain": 0, "single": 1, "half_double": 2,
                     "double": 3, "treble": 4, "fan": 3}.get(dom_type, 1)
        else:
            extra = 1
        return f"{n_stitches + extra}ch"


class CrochetPARADEGenerator:
    """Generates compilable CrochetPARADE code from YOLO detections."""

    def __init__(self, detections, img_h, img_w):
        self.detections = detections
        self.img_h = img_h
        self.img_w = img_w
        self.rows = group_into_rows(detections, img_h)
        self.analyses = []
        self.code_lines = []
        self.warnings = []

    def analyze(self):
        """Analyze all rows."""
        for i, row in enumerate(self.rows):
            analysis = analyze_row(row, i)
            self.analyses.append(analysis)

        # Print summary
        print(f"Detected {len(self.rows)} rows from {len(self.detections)} detections")
        for i, a in enumerate(self.analyses):
            types_str = ", ".join(f"{k}={v}" for k, v in Counter(a.stitch_types).most_common())
            flags = []
            if a.is_chain_row: flags.append("CHAIN_ROW")
            if a.has_fans: flags.append("HAS_FANS")
            if a.is_turning_chain: flags.append("TURNING_CH")
            v_pairs = detect_v_stitches(a.detections)
            if v_pairs: flags.append(f"V_STITCH×{len(v_pairs)}")
            flag_str = f"  [{', '.join(flags)}]" if flags else ""
            print(f"  Row {i}: {len(a.stitch_types)} stitches ({types_str}){flag_str}")

    def generate(self):
        """Generate CrochetPARADE code."""
        if not self.analyses:
            self.analyze()

        lines = []

        # Header comment
        lines.append(f"# CrochetPARADE pattern generated from image detection")
        lines.append(f"# {len(self.rows)} rows, {len(self.detections)} total detections")
        lines.append(f"# Image size: {self.img_w} × {self.img_h}")
        lines.append("")

        # Determine if the first detected row is a foundation chain
        if not self.analyses:
            self.warnings.append("No rows detected")
            self.code_lines = lines
            return "\n".join(lines)

        first = self.analyses[0]
        start_idx = 0

        # If first row is a chain row, use it directly as foundation
        if first.is_chain_row:
            n_chains = len(first.stitch_types)
            lines.append(f"{n_chains}ch,turn")
            start_idx = 1
        else:
            # Generate a foundation chain based on first row stitch count
            foundation = build_foundation_chain(first)
            lines.append(f"{foundation},turn")

        # Generate each subsequent row
        for i in range(start_idx, len(self.analyses)):
            analysis = self.analyses[i]
            prev = self.analyses[i - 1] if i > 0 else None

            row_line = generate_parade_row(analysis, i, len(self.analyses), prev)
            if row_line:
                lines.append(row_line)
            else:
                self.warnings.append(f"Row {i}: empty after conversion")

        self.code_lines = lines

        if self.warnings:
            lines.append("")
            lines.append("# ── Warnings ──")
            for w in self.warnings:
                lines.append(f"# {w}")

        return "\n".join(lines)

    def get_code(self):
        """Return the generated code."""
        if not self.code_lines:
            self.generate()
        return "\n".join(self.code_lines)

    def get_stats(self):
        """Return generation statistics."""
        all_types = []
        for a in self.analyses:
            all_types.extend(a.stitch_types)
        counts = Counter(all_types)
        return {
            "total_detections": len(self.detections),
            "rows": len(self.rows),
            "stitch_counts": dict(counts),
            "has_fans": any(a.has_fans for a in self.analyses),
            "has_v_stitches": any(detect_v_stitches(a.detections)
                                  for a in self.analyses),
            "warnings": self.warnings,
        }


print("CrochetPARADE generator ready.")


CrochetPARADE generator ready.


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# Run YOLO Detection on Image
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO

# Load model
print(f"Loading model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)

# Load image
image = cv.imread(TEST_IMAGE)
if image is None:
    raise FileNotFoundError(f"Cannot read: {TEST_IMAGE}")
img_h, img_w = image.shape[:2]
print(f"Image loaded: {img_w}×{img_h}")

# Run adaptive tiled inference
print("Running detection...")
detections = predict_adaptive(model, image, target_stitch_px=100, conf=0.25)
print(f"Detected {len(detections)} stitches")

# Show class distribution
counts = Counter(d.cls_name for d in detections)
for cls_name, n in counts.most_common():
    print(f"  {cls_name}: {n}")


Loading model: /Users/elevchenko/Documents/DataScience/Crochet/runs/obb/train6/weights/best.pt
Image loaded: 1356×1276
Running detection...
Detected 884 stitches
  half_double: 473
  chain: 251
  single: 108
  treble: 32
  double: 20


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# Generate CrochetPARADE Code
# ═══════════════════════════════════════════════════════════════════════════════

generator = CrochetPARADEGenerator(detections, img_h, img_w)
generator.analyze()

print("\n" + "═" * 60)
print("GENERATED CrochetPARADE CODE")
print("═" * 60 + "\n")

code_output = generator.generate()
print(code_output)

print("\n" + "═" * 60)
stats = generator.get_stats()
print(f"Stats: {stats['rows']} rows, {stats['total_detections']} detections")
if stats['has_fans']:
    print("  → Contains fan/shell stitches (expanded to 5dc)")
if stats['has_v_stitches']:
    print("  → Contains V-stitches (dc,ch,dc)")
if stats['warnings']:
    print(f"  → {len(stats['warnings'])} warnings")


Detected 16 rows from 884 detections
  Row 0: 4 stitches (half_double=2, double=2)
  Row 1: 96 stitches (chain=45, half_double=43, treble=8)
  Row 2: 78 stitches (half_double=54, chain=14, single=9, treble=1)
  Row 3: 31 stitches (half_double=16, chain=10, treble=4, single=1)
  Row 4: 50 stitches (half_double=38, chain=8, single=4)
  Row 5: 72 stitches (half_double=37, chain=21, single=12, treble=2)
  Row 6: 333 stitches (half_double=174, single=80, chain=75, treble=4)
  Row 7: 40 stitches (half_double=29, chain=10, single=1)
  Row 8: 19 stitches (half_double=11, chain=4, treble=4)
  Row 9: 20 stitches (half_double=15, chain=2, treble=2, single=1)
  Row 10: 59 stitches (half_double=42, chain=16, treble=1)
  Row 11: 56 stitches (chain=38, half_double=12, treble=6)
  Row 12: 5 stitches (chain=5)  [CHAIN_ROW, TURNING_CH]
  Row 13: 3 stitches (chain=3)  [CHAIN_ROW, TURNING_CH]
  Row 14: 17 stitches (double=17)
  Row 15: 1 stitches (double=1)

═══════════════════════════════════════════════

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Save Code & Visualization
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Save the CrochetPARADE code to a text file
output_dir = os.path.join(PROJECT_DIR, "output")
os.makedirs(output_dir, exist_ok=True)

code_path = os.path.join(output_dir, "pattern.crochetparade.txt")
with open(code_path, "w") as f:
    f.write(generator.get_code())
print(f"Code saved to: {code_path}")

# ──── Visualization: detections on image with row coloring ────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Left: detection boxes with row coloring
img_rgb = cv.cvtColor(image, cv.COLOR_BGR2RGB)
ax1.imshow(img_rgb)
ax1.set_title("YOLO OBB Detections (colored by row)")

row_colors = plt.cm.Set3(np.linspace(0, 1, max(len(generator.rows), 1)))
for row_idx, row in enumerate(generator.rows):
    color = row_colors[row_idx % len(row_colors)]
    for det in row:
        corners = det.corners
        closed = np.vstack([corners, corners[0]])
        ax1.plot(closed[:, 0], closed[:, 1], color=color, linewidth=1.5)
        cx, cy = corners.mean(axis=0)
        abbr = YOLO_TO_PARADE.get(det.cls_name, "?")
        if abbr is None:
            abbr = det.cls_name[:3]
        ax1.text(cx, cy, abbr, fontsize=6, ha='center', va='center',
                 color='white', fontweight='bold',
                 bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=1))

ax1.axis('off')

# Right: the generated code as formatted text
ax2.set_facecolor('#faf8f4')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_title("Generated CrochetPARADE Code")

code_text = generator.get_code()
# Remove comments for display
display_lines = [l for l in code_text.split("\n") if l.strip() and not l.startswith("#")]
text_block = "\n".join(display_lines)

ax2.text(0.05, 0.95, text_block,
         fontfamily='monospace', fontsize=8, verticalalignment='top',
         transform=ax2.transAxes,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#f0ece4', alpha=0.9))
ax2.axis('off')

plt.tight_layout()
viz_path = os.path.join(output_dir, "parade_codegen_preview.png")
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Visualization saved to: {viz_path}")


Code saved to: /Users/elevchenko/Documents/DataScience/Crochet/output/pattern.crochetparade.txt


/var/folders/b5/4hk8xrkx3x50b85sdqdmy76w0000gn/T/ipykernel_62318/1927920715.py:61: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Visualization saved to: /Users/elevchenko/Documents/DataScience/Crochet/output/parade_codegen_preview.png


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# CrochetPARADE Code Validation
# ═══════════════════════════════════════════════════════════════════════════════
# Check the generated code against known CrochetPARADE grammar rules.

VALID_STITCHES = {
    "ch", "sc", "hdc", "dc", "tr", "dtr", "trtr", "ss", "sk",
    "rsc", "turn",
    "scbl", "scfl", "dcbl", "dcfl", "hdcbl", "hdcfl", "trbl", "trfl",
    "bpdc", "fpdc", "bpsc", "fpsc", "bphdc", "fphdc",
    "longsc", "longdc", "longtr",
    "picot3",
    "ring", "tie_up",
}

# Stitch patterns that include numeric prefixes/suffixes
import re

STITCH_PATTERN = re.compile(
    r'^(\d+\*?)?'                        # optional multiplier: 5, 5*
    r'(\[.*\]|'                           # bracketed group [...]
    r'(?:sc|hdc|dc|tr|dtr|trtr|ss|ch|sk|rsc|'
    r'scbl|scfl|dcbl|dcfl|hdcbl|hdcfl|trbl|trfl|'
    r'bpdc|fpdc|bpsc|fpsc|bphdc|fphdc|'
    r'longsc|longdc|longtr|'
    r'picot3|ring|tie_up|turn)'           # stitch name
    r'(\d+(?:inc|tog))?'                  # optional inc/tog suffix
    r')$'
)


def validate_parade_code(code_str):
    """Validate CrochetPARADE code and return (is_valid, errors, warnings)."""
    errors = []
    warnings = []

    lines = code_str.strip().split("\n")
    non_comment_lines = [l.strip() for l in lines
                         if l.strip() and not l.strip().startswith("#")]

    if not non_comment_lines:
        errors.append("No pattern lines found")
        return False, errors, warnings

    for line_num, line in enumerate(non_comment_lines):
        # Handle DEF lines
        if line.startswith("DEF:"):
            continue
        # Handle COLOR lines
        if line.startswith("COLOR:"):
            continue
        # Handle DOT lines
        if line.startswith("DOT:"):
            continue

        # Split line into stitch tokens, respecting brackets
        tokens = []
        depth = 0
        current = ""
        for char in line:
            if char == "[":
                depth += 1
                current += char
            elif char == "]":
                depth -= 1
                current += char
            elif char == "," and depth == 0:
                if current.strip():
                    tokens.append(current.strip())
                current = ""
            else:
                current += char
        if current.strip():
            tokens.append(current.strip())

        for token in tokens:
            # Check if 'turn' is not last
            if token == "turn" and token != tokens[-1]:
                errors.append(f"Line {line_num + 1}: 'turn' must be the last element")

            # Check for valid multiplier+stitch pattern
            # Strip attachment (@...) and label (.X) for validation
            clean = re.sub(r'@.*$', '', token)
            clean = re.sub(r'\.\w+$', '', clean)

            if clean == "turn":
                continue

            # Handle bracketed groups like 4*[sc,dc]
            bracket_match = re.match(r'^(\d+\*?)?\[(.+)\]$', clean)
            if bracket_match:
                inner = bracket_match.group(2)
                inner_tokens = [t.strip() for t in inner.split(",")]
                for it in inner_tokens:
                    it_clean = re.sub(r'@.*$', '', it)
                    it_clean = re.sub(r'\.\w+$', '', it_clean)
                    m = re.match(r'^(\d+)?(\w+)$', it_clean)
                    if m:
                        stitch = m.group(2)
                        if stitch not in VALID_STITCHES:
                            warnings.append(
                                f"Line {line_num+1}: unknown stitch '{stitch}' in bracket group")
                continue

            # Validate stitch token
            if not STITCH_PATTERN.match(clean):
                # Check if it's just a number+stitch (e.g., "5dc", "3ch")
                m = re.match(r'^(\d+)(\w+)$', clean)
                if m:
                    num, stitch = m.groups()
                    if stitch not in VALID_STITCHES and \
                       not re.match(r'(sc|dc|hdc|tr)\d+(inc|tog)', stitch):
                        warnings.append(
                            f"Line {line_num + 1}: unknown stitch '{stitch}' in '{token}'")
                elif clean not in VALID_STITCHES:
                    warnings.append(
                        f"Line {line_num + 1}: unrecognized token '{token}'")

    # Check first line starts with chain foundation
    first_line = non_comment_lines[0] if non_comment_lines else ""
    if not re.match(r'^\d*ch', first_line):
        warnings.append(
            "First line should start with a chain foundation (e.g., '20ch,turn')")

    is_valid = len(errors) == 0
    return is_valid, errors, warnings


# Run validation
code_text = generator.get_code()
is_valid, errors, warnings = validate_parade_code(code_text)

print("CrochetPARADE Validation Results")
print("=" * 40)
if is_valid:
    print("✓ Code is valid (no structural errors)")
else:
    print("✗ Code has errors:")
    for e in errors:
        print(f"  ERROR: {e}")

if warnings:
    print(f"\n{len(warnings)} warnings:")
    for w in warnings:
        print(f"  WARN: {w}")
else:
    print("No warnings.")

print("\n── How to compile ──")
print("1. Go to https://www.crochetparade.org/")
print("2. Paste the code into the editor")
print("3. Click 'Compile' (or press Ctrl+Enter)")
print("4. The 3D model and crochet chart will render automatically")
print(f"\nCode file: {code_path}")


CrochetPARADE Validation Results
✓ Code is valid (no structural errors)
No warnings.

── How to compile ──
1. Go to https://www.crochetparade.org/
2. Paste the code into the editor
3. Click 'Compile' (or press Ctrl+Enter)
4. The 3D model and crochet chart will render automatically

Code file: /Users/elevchenko/Documents/DataScience/Crochet/output/pattern.crochetparade.txt


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# Utility: Generate from Detections JSON
# ═══════════════════════════════════════════════════════════════════════════════
# This function can be called from the MCP tool (crochet_tool.py) to generate
# CrochetPARADE code directly from the detection JSON output.

def generate_parade_from_detections_json(det_json, img_w, img_h):
    """Generate CrochetPARADE code from the JSON detection format
    produced by crochet_tool.py's analyze_crochet_chart().

    Args:
        det_json: list of detection dicts with keys:
            class, abbreviation, confidence, row, center_x, center_y,
            width, height, angle_deg, corners
        img_w, img_h: image dimensions

    Returns:
        dict with keys: code, stats, is_valid, errors, warnings
    """
    # Convert JSON detections back to Detection objects
    detections = []
    for d in det_json:
        corners = np.array(d["corners"], dtype=np.float32)
        cls_name = d["class"]
        cls_id = CLASS_MAP.get(cls_name, -1)
        detections.append(Detection(
            corners=corners,
            cls_id=cls_id,
            cls_name=cls_name,
            confidence=d["confidence"],
        ))

    # Generate
    gen = CrochetPARADEGenerator(detections, img_h, img_w)
    code_str = gen.generate()
    stats = gen.get_stats()

    # Validate
    is_valid, errors, warnings = validate_parade_code(code_str)

    return {
        "code": code_str,
        "stats": stats,
        "is_valid": is_valid,
        "errors": errors,
        "warnings": warnings,
    }


# Quick test with current detections
print("Testing generate_parade_from_detections_json()...")
test_json = []
for det in detections:
    g = det_geometry(det)
    test_json.append({
        "class": det.cls_name,
        "abbreviation": YOLO_TO_PARADE.get(det.cls_name, "?"),
        "confidence": float(det.confidence),
        "row": 0,
        "center_x": g["cx"],
        "center_y": g["cy"],
        "width": g["w"],
        "height": g["h"],
        "angle_deg": g["angle"],
        "corners": det.corners.tolist(),
    })

result = generate_parade_from_detections_json(test_json, img_w, img_h)
print(f"  Valid: {result['is_valid']}")
print(f"  Rows: {result['stats']['rows']}")
print(f"  Code length: {len(result['code'])} chars")
print("\nDone!")


Testing generate_parade_from_detections_json()...
Detected 16 rows from 884 detections
  Row 0: 4 stitches (half_double=2, double=2)
  Row 1: 96 stitches (chain=45, half_double=43, treble=8)
  Row 2: 78 stitches (half_double=54, chain=14, single=9, treble=1)
  Row 3: 31 stitches (half_double=16, chain=10, treble=4, single=1)
  Row 4: 50 stitches (half_double=38, chain=8, single=4)
  Row 5: 72 stitches (half_double=37, chain=21, single=12, treble=2)
  Row 6: 333 stitches (half_double=174, single=80, chain=75, treble=4)
  Row 7: 40 stitches (half_double=29, chain=10, single=1)
  Row 8: 19 stitches (half_double=11, chain=4, treble=4)
  Row 9: 20 stitches (half_double=15, chain=2, treble=2, single=1)
  Row 10: 59 stitches (half_double=42, chain=16, treble=1)
  Row 11: 56 stitches (chain=38, half_double=12, treble=6)
  Row 12: 5 stitches (chain=5)  [CHAIN_ROW, TURNING_CH]
  Row 13: 3 stitches (chain=3)  [CHAIN_ROW, TURNING_CH]
  Row 14: 17 stitches (double=17)
  Row 15: 1 stitches (double=1